# Pré-processamento das solicitações GERCON (executante Porto Alegre, 2018-2025)

Entrada: `dados/brutos/gercon_solicitacoes_executante_porto_alegre_2018_2025.parquet`

Saída: `dados/processados/gercon_solicitacoes_executante_porto_alegre_2018_2025_pre.parquet`

Etapas:
1. leitura do parquet bruto (todas as colunas vêm como string);
2. remoção de espaços nas pontas e vazios viram nulo;
3. conversão das colunas de data (formatos mistos no arquivo);
4. conversão das colunas numéricas (`diasfilaespera`, `idade`);
5. identificadores e códigos mantidos como texto (zeros à esquerda);
6. filtros de recorte;
7. gravação do parquet.

In [243]:
import os
import pandas as pd

RAIZ = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'andre' else os.getcwd()
ENTRADA = os.path.join(RAIZ, 'dados', 'brutos', 'gercon_solicitacoes_executante_porto_alegre_2018_2025.parquet')
SAIDA = os.path.join(RAIZ, 'dados', 'processados', 'gercon_solicitacoes_executante_porto_alegre_2018_2025_pre.parquet')

COLUNAS_DATA = ['data_extracao', 'data_solicitacao', 'datahora_consulta']
COLUNAS_NUMERICAS = ['diasfilaespera', 'idade']
COLUNAS_TEXTO_ID = ['idsolicitacao', 'idpaciente', 'cnesexecutante', 'cnes_solicitante',
                    'COD_IBGE_MUNICIPIO_EXECUTANTE_7D', 'COD_IBGE_MUNICIPIO_EXECUTANTE_6D']

FILTRO_MUNICIPIO_SOLICITANTE = 'PORTO ALEGRE'
FILTRO_TIPO_UNIDADE = 'CENTRO DE SAUDE/UNIDADE BASICA'
FILTRO_COMPLEXIDADE = 'MEDIA'
FILTRO_SITUACAO_AGENDA = ['REALIZADA', 'FALTANTE']

print('Entrada:', ENTRADA)
print('Saída:', SAIDA)

Entrada: /home/andrebueno/Google_drive/01_Doutorado/Tese/banco/gercon/dados/brutos/gercon_solicitacoes_executante_porto_alegre_2018_2025.parquet
Saída: /home/andrebueno/Google_drive/01_Doutorado/Tese/banco/gercon/dados/processados/gercon_solicitacoes_executante_porto_alegre_2018_2025_pre.parquet


## 1. Leitura

In [244]:
df = pd.read_parquet(ENTRADA)
print(f'{df.shape[0]:,} linhas x {df.shape[1]} colunas')
df.head()

2,246,275 linhas x 33 colunas


,data_extracao,idsolicitacao,situacaosolicitacao,data_solicitacao,corregulador,diasfilaespera,complexidade,especialidade,especialidademae,cid,...,centralregulacao,centralregulacaoorigem,tipoconsulta,coordenadoriasolicitante,regiaosaudesolicitante,possuiunidadeindicada,COD_IBGE_MUNICIPIO_EXECUTANTE_7D,COD_IBGE_MUNICIPIO_EXECUTANTE_6D,UF_EXECUTANTE,ARQUIVO_ORIGEM
0,2025-01-07 00:00:00.000,747495,REALIZADA,2018-02-20 00:00:00.000,AMARELO,17,MEDIA,CIRURGIA BUCOMAXILOFACIAL CEO,CIRURGIA BUCOMAXILOFACIAL,K011,...,REGULACAO AMBULATORIAL,REGULACAO AMBULATORIAL,PRIMEIRA,1ª CRS,REGIÃO DE SAÚDE 10,SIM,4314902,431490,RS,gercon_solicitacoes_2018.csv
1,2025-01-07 00:00:00.000,756290,REALIZADA,2018-02-28 00:00:00.000,AZUL,14,MEDIA,CIRURGIA BUCOMAXILOFACIAL CEO,CIRURGIA BUCOMAXILOFACIAL,K021,...,REGULACAO AMBULATORIAL,REGULACAO AMBULATORIAL,PRIMEIRA,1ª CRS,REGIÃO DE SAÚDE 10,SIM,4314902,431490,RS,gercon_solicitacoes_2018.csv
2,2025-01-07 00:00:00.000,806777,REALIZADA,2018-04-12 00:00:00.000,AZUL,1,MEDIA,CIRURGIA BUCOMAXILOFACIAL CEO,CIRURGIA BUCOMAXILOFACIAL,K011,...,REGULACAO AMBULATORIAL,REGULACAO AMBULATORIAL,PRIMEIRA,1ª CRS,REGIÃO DE SAÚDE 10,NÃO,4314902,431490,RS,gercon_solicitacoes_2018.csv
3,2025-01-07 00:00:00.000,730573,REALIZADA,2018-02-01 00:00:00.000,LARANJA,5,MEDIA,PERIODONTIA,ODONTO PERIODONTIA,K061,...,REGULACAO AMBULATORIAL,REGULACAO AMBULATORIAL,PRIMEIRA,1ª CRS,REGIÃO DE SAÚDE 10,NÃO,4314902,431490,RS,gercon_solicitacoes_2018.csv
4,2025-01-07 00:00:00.000,744023,ENCERRADA,2018-02-16 00:00:00.000,NENHUM,0,MEDIA,PERIODONTIA,ODONTO PERIODONTIA,K051,...,REGULACAO AMBULATORIAL,REGULACAO AMBULATORIAL,PRIMEIRA,1ª CRS,REGIÃO DE SAÚDE 10,NÃO,4314902,431490,RS,gercon_solicitacoes_2018.csv


## 2. Limpeza de texto

In [245]:
colunas_texto = df.select_dtypes(include=['object', 'string']).columns
for c in colunas_texto:
    df[c] = df[c].str.strip().replace('', pd.NA)

## 3. Datas e numéricos

In [246]:
for c in COLUNAS_DATA:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], format='mixed', errors='coerce')

for c in COLUNAS_NUMERICAS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

for c in COLUNAS_TEXTO_ID:
    if c in df.columns:
        df[c] = df[c].astype('string')

df[COLUNAS_DATA + COLUNAS_NUMERICAS].dtypes

data_extracao        datetime64[us]
data_solicitacao     datetime64[us]
datahora_consulta    datetime64[us]
diasfilaespera                int64
idade                         int64
dtype: object

## 4. Filtros de recorte

- `municipiosolicitante` = PORTO ALEGRE
- `tipounidade` = CENTRO DE SAUDE/UNIDADE BASICA
- `complexidade` = MEDIA
- `datahora_consulta` não nulo
- `situacaoagenda` em REALIZADA e FALTANTE

In [247]:
n0 = len(df)
mascara = (
    (df['municipiosolicitante'] == FILTRO_MUNICIPIO_SOLICITANTE)
    & (df['tipounidade'] == FILTRO_TIPO_UNIDADE)
    & (df['complexidade'] == FILTRO_COMPLEXIDADE)
    & df['datahora_consulta'].notna()
    & df['situacaoagenda'].isin(FILTRO_SITUACAO_AGENDA)
)
df = df[mascara].reset_index(drop=True)
print(f'{n0:,} -> {len(df):,} linhas')

2,246,275 -> 994,058 linhas


# Merge CNES
Site - https://cnes.datasus.gov.br/pages/downloads/arquivosBaseDados.jsp

Arquivo - BASE_DE_DADOS_CNES_202609.ZIP


In [248]:
import pandas as pd

kw = dict(sep=";", encoding="latin-1", dtype=str)

estab = pd.read_csv("tbEstabelecimento202609.csv", **kw)
nat_jur = pd.read_csv("tbNaturezaJuridica202609.csv", **kw)

estab.shape, nat_jur.shape

((640122, 56), (100, 2))

In [249]:
estab_nj = estab.merge(nat_jur, on="CO_NATUREZA_JUR", how="left", validate="m:1")

In [250]:
df = df.merge(
    estab_nj[["CO_CNES", "NO_RAZAO_SOCIAL", "DS_NATUREZA_JUR"]],
    left_on="cnesexecutante",
    right_on="CO_CNES",
    how="left",
    validate="m:1",
).drop(columns="CO_CNES")

In [251]:
# filtrar por DS_NATUREZA_JUR igual a MUNICIPIO 
df = df[df["DS_NATUREZA_JUR"] == "MUNICIPIO"].reset_index(drop=True)

In [252]:
# Valores unicos de NO_RAZAO_SOCIAL, com a frequencia % com 2 digitos   
df["NO_RAZAO_SOCIAL"].value_counts(normalize=True).round(2)

NO_RAZAO_SOCIAL
MUNICIPIO DE PORTO ALEGRE                              0.48
PREFEITURA MUNICIPAL DE PORTO ALEGRE                   0.45
HOSPITAL MATERNO INFANTIL PRESIDENTE VARGAS            0.07
HOSPITAL NOSSA SENHORA DA CONCEICAO                    0.00
PRONTO ATENDIMENTO CRUZEIRO DO SUL                     0.00
PREFEITURA MUNICIPAL DE IMBE                           0.00
CER CENTRO DE REABILITACAO FISICA AUDITIVA E VISUAL    0.00
MUNICIPIO DE CAPAO DA CANOA                            0.00
Name: proportion, dtype: float64

In [253]:
df["NO_RAZAO_SOCIAL"].unique()

<ArrowStringArray>
[               'PREFEITURA MUNICIPAL DE PORTO ALEGRE',
                           'MUNICIPIO DE PORTO ALEGRE',
         'HOSPITAL MATERNO INFANTIL PRESIDENTE VARGAS',
                  'PRONTO ATENDIMENTO CRUZEIRO DO SUL',
                        'PREFEITURA MUNICIPAL DE IMBE',
 'CER CENTRO DE REABILITACAO FISICA AUDITIVA E VISUAL',
                         'MUNICIPIO DE CAPAO DA CANOA',
                 'HOSPITAL NOSSA SENHORA DA CONCEICAO']
Length: 8, dtype: str

In [254]:
# Quero filtra apenas os NO_RAZAO_SOCIAL PREFEITURA MUNICIPAL DE PORTO ALEGRE, MUNICIPIO DE PORTO ALEGRE, HOSPITAL MATERNO INFANTIL PRESIDENTE VARGAS e PRONTO ATENDIMENTO CRUZEIRO DO SUL
df = df[df["NO_RAZAO_SOCIAL"].isin(["PREFEITURA MUNICIPAL DE PORTO ALEGRE", "MUNICIPIO DE PORTO ALEGRE", "HOSPITAL MATERNO INFANTIL PRESIDENTE VARGAS", "PRONTO ATENDIMENTO CRUZEIRO DO SUL"])].reset_index(drop=True)

In [255]:
# Quantos cnesexecutantes unicos temos?
df["cnesexecutante"].nunique()

111

# Merge CENSO

https://github.com/abueno78/censo-territorios-aps

outputs/tabelas_unidades_saude_v2.xlsx

variaveis Favela/Comunidade,taxa_alf_geral,renda_media_winsor,coordenadoria_saude e distrito_sanitario

In [256]:
import io
import pandas as pd
import requests

URL_CENSO = (
    "https://raw.githubusercontent.com/abueno78/censo-territorios-aps/"
    "main/outputs/tabelas_unidades_saude_v2.xlsx"
)
ABA_CENSO = "Indicadores_Completos_v2"
CHAVES_CENSO = ["CNES", "Unidade de Saúde"]
COLUNAS_CENSO = {
    "Favela/Comunidade": "favela_comunidade",
    "taxa_alf_geral": "taxa_alf_geral",
    "renda_media_winsor": "renda_media_winsor",
    "coordenadoria_saude": "coordenadoria_saude",
    "distrito_sanitario": "distrito_sanitario",
}

resp = requests.get(URL_CENSO, timeout=120)
resp.raise_for_status()

censo = (
    pd.read_excel(
        io.BytesIO(resp.content),
        sheet_name=ABA_CENSO,
        usecols=CHAVES_CENSO + list(COLUNAS_CENSO),
    )
    .rename(columns=COLUNAS_CENSO)
)
censo.head()

,Unidade de Saúde,favela_comunidade,taxa_alf_geral,CNES,coordenadoria_saude,distrito_sanitario,renda_media_winsor
0,CF Belém Novo,3.972632,96.932851,2264471.0,SUL,EXTREMO SUL,3860.293690
1,CF Campo da Tuca,58.768085,97.047483,8006865.0,LESTE,PARTENON,2532.535305
2,CF Diretor Pestana,14.748452,98.196073,2264374.0,NORTE,Fechada para reunião 4° quinta-feira do mês 14...,3970.817382
3,CF IAPI,1.459531,99.276839,6883303.0,NORTE,NOROESTE,8355.090506
4,CF José Mauro Ceratti Lopes,14.905920,94.976008,2264404.0,SUL,RESTINGA,1700.161961


In [257]:
# padronizar a CNES- é uma variavel categorica com 7 digitos com zeros à esquerda - 2264471.00 deve ficar  2264471
censo["CNES"] = censo["CNES"].astype(str)

In [258]:
#remover o .0 no final dos valores da variavel CNES
censo["CNES"] = censo["CNES"].str.replace(r"\.0$", "", regex=True)

In [259]:
# padronizar em 7 digitos com zeros à esquerda
censo["CNES"] = censo["CNES"].str.zfill(7)

In [260]:
censo.head()

,Unidade de Saúde,favela_comunidade,taxa_alf_geral,CNES,coordenadoria_saude,distrito_sanitario,renda_media_winsor
0,CF Belém Novo,3.972632,96.932851,2264471,SUL,EXTREMO SUL,3860.293690
1,CF Campo da Tuca,58.768085,97.047483,8006865,LESTE,PARTENON,2532.535305
2,CF Diretor Pestana,14.748452,98.196073,2264374,NORTE,Fechada para reunião 4° quinta-feira do mês 14...,3970.817382
3,CF IAPI,1.459531,99.276839,6883303,NORTE,NOROESTE,8355.090506
4,CF José Mauro Ceratti Lopes,14.905920,94.976008,2264404,SUL,RESTINGA,1700.161961


In [261]:
censo["distrito_sanitario"].unique()

<ArrowStringArray>
[                                           'EXTREMO SUL',
                                               'PARTENON',
 'Fechada para reunião 4° quinta-feira do mês 14h as 16h',
                                               'NOROESTE',
                                               'RESTINGA',
                                               'CRUZEIRO',
                                                 'CENTRO',
                                                  'LESTE',
                                     'HUMAITA/NAVEGANTES',
                                                 'GLORIA',
                                                    'SUL',
                                                      nan,
                                             'CENTRO SUL',
                                                  'NORTE',
                                               'NORDESTE',
                                          'EIXO BALTAZAR',
                                     

variavel distrito\_sanitario tem 1 categoria com problema. "Fechada para reunião 4° quinta-feira do mês 14h as 16h". Vamos ver quantas unidades são:
variavel distrito\_sanitario tem 1 categoria com problema. "Fechada para reunião 4° quinta-feira do mês 14h as 16h". Vamos ver quantas unidades são: 

In [262]:
CNES_PESTANA = "2264374"
DISTRITO_PESTANA = "HUMAITA/NAVEGANTES"

# corrige apenas no censo; o df herda o valor via merge
censo.loc[censo["CNES"] == CNES_PESTANA, "distrito_sanitario"] = DISTRITO_PESTANA

censo["distrito_sanitario"].value_counts(dropna=False)

distrito_sanitario
LESTE                 15
NORTE                 14
PARTENON              11
CENTRO SUL            11
EIXO BALTAZAR         11
LOMBA DO PINHEIRO      9
RESTINGA               8
GLORIA                 8
NORDESTE               8
SUL                    7
EXTREMO SUL            5
HUMAITA/NAVEGANTES     5
NOROESTE               5
CRUZEIRO               5
CRISTAL                5
CENTRO                 3
ILHAS                  2
NaN                    1
Name: count, dtype: int64

In [263]:
censo["distrito_sanitario"].unique()

<ArrowStringArray>
[       'EXTREMO SUL',           'PARTENON', 'HUMAITA/NAVEGANTES',           'NOROESTE',
           'RESTINGA',           'CRUZEIRO',             'CENTRO',              'LESTE',
             'GLORIA',                'SUL',                  nan,         'CENTRO SUL',
              'NORTE',           'NORDESTE',      'EIXO BALTAZAR',            'CRISTAL',
  'LOMBA DO PINHEIRO',              'ILHAS']
Length: 18, dtype: str

In [184]:
# df variavel cnes_solicitante unicos e seus nomes de unidades solicitantes
vars = df[["cnes_solicitante", "unidadesolicitante"]].drop_duplicates()

In [264]:
# padronizar cnes_solicitante: texto, sem ".0", 7 dígitos com zeros à esquerda
df["cnes_solicitante"] = (
    df["cnes_solicitante"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(7)
)


In [265]:
# realizar o Merge com df cnes_solicitante com CNES do censo
df = df.merge(
    censo,
    left_on="cnes_solicitante",
    right_on="CNES",
    how="left",
    validate="m:1",
).drop(columns="CNES")

In [267]:
PADRAO_APS = "UNIDADE DE SAUDE|ESF NUCLEO ESPERANCA II"

sem_censo = df["taxa_alf_geral"].isna()
nao_aps = ~df["unidadesolicitante"].str.contains(PADRAO_APS, na=False)

# 1) sem Censo e fora do padrão de APS
n0 = len(df)
df = df.loc[~(sem_censo & nao_aps)].copy()
print(f"Removidas (sem Censo e não APS): {n0 - len(df):,} -> {len(df):,} linhas")

# 2) APS restantes sem Censo
pct_na = df["taxa_alf_geral"].isna().mean() * 100
print(f"% restante sem Censo: {pct_na:.2f}%")

df = df.loc[df["taxa_alf_geral"].notna()].copy()
print(f"Banco final: {df.shape[0]:,} linhas x {df.shape[1]} colunas")

Removidas (sem Censo e não APS): 552 -> 364,173 linhas
% restante sem Censo: 0.90%
Banco final: 360,908 linhas x 41 colunas


In [268]:
df.info()

<class 'pandas.DataFrame'>
Index: 360908 entries, 0 to 364724
Data columns (total 41 columns):
 #   Column                            Non-Null Count   Dtype         
---  ------                            --------------   -----         
 0   data_extracao                     360908 non-null  datetime64[us]
 1   idsolicitacao                     360908 non-null  string        
 2   situacaosolicitacao               360908 non-null  str           
 3   data_solicitacao                  360908 non-null  datetime64[us]
 4   corregulador                      360878 non-null  str           
 5   diasfilaespera                    360908 non-null  int64         
 6   complexidade                      360908 non-null  str           
 7   especialidade                     360908 non-null  str           
 8   especialidademae                  360908 non-null  str           
 9   cid                               360908 non-null  str           
 10  idpaciente                        360908 non-nul

# Merge Google Maps 
Calcular as distancias entre as unidades solicitante e executante

script no arquivo - distancias.r

Distância e transporte
variaveis - distancia_km, tempo_caminhando, distancia_km_trans_publico, tempo_desloc_trans_publico, quant_bus, baldiacoes


In [ ]:
import pandas as pd

CHAVES = ["id_endereco_cnes_solicitante", "id_endereco_cnesexecutante"]
COLS_DIST = [
    "status", "distancia_km", "tempo_caminhando",
    "distancia_km_trans_publico", "tempo_desloc_trans_publico",
    "quant_bus", "baldiacoes",
]

# CNES -> chave de endereço
mapa_cnes = (
    pd.read_csv(
        "dados/distancias/enderecos_cnes_detalhe.csv",
        usecols=["CO_CNES", "chave_endereco"],
        dtype=str,
    )
    .drop_duplicates()
)
mapa_cnes["CO_CNES"] = mapa_cnes["CO_CNES"].str.zfill(7)
assert mapa_cnes["CO_CNES"].is_unique

# Base de distâncias (uma linha por par de endereços)
dist = pd.read_parquet(
    "dados/distancias/base_distancias_onibus.parquet",
    columns=CHAVES + COLS_DIST,
)
assert not dist.duplicated(CHAVES).any()

# Mesmo tratamento do R: sem_rota sem valores = mesmo endereço -> zeros
cols_zero = [c for c in COLS_DIST if c != "status"]
sem_rota_na = (dist["status"] == "sem_rota") & dist["distancia_km"].isna()
dist.loc[sem_rota_na, cols_zero] = 0

# Merge: CNES -> endereço -> distâncias
n0 = len(df)
df = (
    df.assign(
        cnes_solicitante=lambda x: x["cnes_solicitante"].astype(str).str.zfill(7),
        cnesexecutante=lambda x: x["cnesexecutante"].astype(str).str.zfill(7),
    )
    .merge(
        mapa_cnes.rename(columns={"CO_CNES": "cnes_solicitante",
                                  "chave_endereco": "id_endereco_cnes_solicitante"}),
        on="cnes_solicitante", how="left",
    )
    .merge(
        mapa_cnes.rename(columns={"CO_CNES": "cnesexecutante",
                                  "chave_endereco": "id_endereco_cnesexecutante"}),
        on="cnesexecutante", how="left",
    )
    .merge(dist, on=CHAVES, how="left", validate="many_to_one")
    .drop(columns=CHAVES)
)



# Processamento de novas informações relacionada ao paciente

A literatura indica que pacientes que faltaram a uma consulta têm maior probabilidade de faltar novamente. Para capturar esse efeito sem **vazamento de dados**, cada linha recebe apenas informações do histórico do paciente anteriores ao momento do agendamento (`t0`). A própria consulta e consultas futuras nunca entram no cálculo.

## Variáveis auxiliares
- **`rid`**: identificador único da linha, usado para juntar os históricos de volta ao banco.
- **`y`**: desfecho da consulta (1 = faltou, 0 = não faltou).
- **`t0`**: data do agendamento, estimada como data da solicitação + dias na fila de espera. Define o que o paciente "já tinha vivido" naquele momento.
- **`cid_3dig`**: CID reduzido aos 3 primeiros caracteres (categoria da doença).
- **`motivo`**: combinação de especialidade-mãe e CID de 3 dígitos, que representa o motivo do encaminhamento.

## Histórico geral do paciente
- **`n_consultas_previas`**: número de consultas anteriores do paciente.
- **`n_faltas_previas`**: número de faltas nessas consultas anteriores.
- **`primeira_vez`**: 1 se o paciente não tem nenhuma consulta anterior.
- **`dias_desde_ultima_falta`**: dias entre a última falta anterior e o agendamento (vazio se nunca faltou).
- **`falta_recente`**: versão categórica da anterior: `nunca_faltou`, `ate_90d`, `91_365d` ou `mais_de_365d`.

## Histórico específico
O mesmo raciocínio, restrito a consultas anteriores na mesma especialidade (`_especialidademae`) ou com o mesmo motivo (`_motivo`):
- **`n_consultas_previas_especialidademae`** e **`n_consultas_previas_motivo`**: número de consultas anteriores.
- **`n_faltas_previas_especialidademae`** e **`n_faltas_previas_motivo`**: número de faltas nessas consultas.
- **`ultima_previa_mesma_especialidademae`** e **`ultima_previa_mesmo_motivo`**: desfecho da consulta anterior mais recente (`sem_previa`, `realizada` ou `faltante`).

## Tempo e calendário
- **`dias_ate_consulta`**: dias entre a solicitação e a consulta.
- **`antecedencia_agendamento`**: dias entre o agendamento (`t0`) e a consulta, ou seja, `dias_ate_consulta - diasfilaespera`.
- **`dia_semana`** e **`mes_consulta`**: dia da semana e mês da consulta.

**Atenção:** `dias_ate_consulta` é a soma de `diasfilaespera` e `antecedencia_agendamento`. No modelo, use apenas uma das parametrizações (`dias_ate_consulta` ou `diasfilaespera` + `antecedencia_agendamento`) para evitar colinearidade perfeita.

In [270]:
import numpy as np

# Distribuição de faltas por paciente (análise descritiva; não gera variável no banco)
ROTULOS_FALTAS = ["Nunca faltou", "1 falta", "2 faltas", "3 faltas", "Mais que 3 faltas"]

faltas_pac = (
    (df["situacaoagenda"] == "FALTANTE")
    .groupby(df["idpaciente"])
    .sum()
)
grupo = pd.cut(faltas_pac, bins=[-1, 0, 1, 2, 3, np.inf], labels=ROTULOS_FALTAS)

tab = grupo.value_counts().reindex(ROTULOS_FALTAS).rename("n").to_frame()
tab["%"] = (tab["n"] / tab["n"].sum() * 100).round(1)
tab.loc["Total"] = [tab["n"].sum(), 100.0]
tab

,n,%
situacaoagenda,,
Nunca faltou,170706,71.6
1 falta,58355,24.5
2 faltas,7529,3.2
3 faltas,1332,0.6
Mais que 3 faltas,433,0.2
Total,238355,100.0


In [ ]:
# --- 0. Variáveis auxiliares -------------------------------------------------
df = df.reset_index(drop=True)

# rid: identificador único da linha (chave para juntar os históricos de volta)
df["rid"] = np.arange(len(df))

# y: desfecho da consulta (1 = faltou, 0 = não faltou)
df["y"] = (df["situacaoagenda"] == "FALTANTE").astype("int8")

# t0: data do agendamento = data da solicitação + dias na fila (negativos tratados como 0).
# Só consultas ocorridas antes de t0 são "conhecidas" -> evita vazamento de dados.
df["t0"] = df["data_solicitacao"] + pd.to_timedelta(
    df["diasfilaespera"].clip(lower=0), unit="D"
)

# cid_3dig: CID com 3 caracteres (categoria da doença)
df["cid_3dig"] = df["cid"].str[:3]

# cid_1dig: CID com 1 caractere (categoria da doença)
df["cid_1dig"] = df["cid_3dig"].str[0]

# motivo: especialidade-mãe + CID de 3 dígitos (motivo do encaminhamento)
df["motivo"] = df["especialidademae"].astype("string") + "|" + df["cid_3dig"].astype("string")

# --- 1. Pares (linha-alvo x consulta anterior do mesmo paciente) -------------
contagem = df["idpaciente"].value_counts()
multi = contagem[contagem >= 2].index
sub = df[df["idpaciente"].isin(multi)]

alvos = sub[["idpaciente", "rid", "t0", "especialidademae", "motivo"]]
eventos = sub[
    ["idpaciente", "rid", "datahora_consulta", "y", "especialidademae", "motivo"]
].rename(
    columns={
        "rid": "rid_ev",
        "datahora_consulta": "data_ev",
        "y": "y_ev",
        "especialidademae": "esp_ev",
        "motivo": "motivo_ev",
    }
)

pares = alvos.merge(eventos, on="idpaciente", how="inner")
# nunca a própria linha; só consultas anteriores ao agendamento
pares = pares[(pares["rid"] != pares["rid_ev"]) & (pares["data_ev"] < pares["t0"])].copy()
pares["mesma_esp"] = pares["especialidademae"] == pares["esp_ev"]
pares["mesmo_motivo"] = pares["motivo"] == pares["motivo_ev"]
pares = pares.sort_values(["rid", "data_ev", "rid_ev"])

# --- 2. Agregações -----------------------------------------------------------
hist_geral = pares.groupby("rid").agg(
    n_consultas_previas=("y_ev", "size"),
    n_faltas_previas=("y_ev", "sum"),
)
hist_geral["ultima_falta"] = pares[pares["y_ev"] == 1].groupby("rid")["data_ev"].max()


def resumo_ultima(p, sufixo):
    """Contagem, faltas e desfecho da consulta anterior mais recente por rid."""
    g = p.groupby("rid")
    return pd.DataFrame(
        {
            f"n_consultas_previas_{sufixo}": g["y_ev"].size(),
            f"n_faltas_previas_{sufixo}": g["y_ev"].sum(),
            f"ultimo_y_{sufixo}": g["y_ev"].last(),  # pares já ordenados por data
        }
    )


hist_esp = resumo_ultima(pares[pares["mesma_esp"]], "especialidademae")
hist_motivo = resumo_ultima(pares[pares["mesmo_motivo"]], "motivo")

# --- 3. Variáveis de histórico -----------------------------------------------
df = df.drop(
    columns=[
        *hist_geral.columns, *hist_esp.columns, *hist_motivo.columns,
    ],
    errors="ignore",
)
df = df.join(hist_geral, on="rid").join(hist_esp, on="rid").join(hist_motivo, on="rid")

# n_consultas_previas*: nº de consultas anteriores do paciente (geral / mesma especialidade / mesmo motivo)
# n_faltas_previas*: nº de faltas nessas consultas anteriores (0 se não há histórico)
cols_int = [
    "n_consultas_previas", "n_faltas_previas",
    "n_consultas_previas_especialidademae", "n_faltas_previas_especialidademae",
    "n_consultas_previas_motivo", "n_faltas_previas_motivo",
]
df[cols_int] = df[cols_int].fillna(0).astype("int32")

# primeira_vez: 1 se o paciente não tem nenhuma consulta anterior
df["primeira_vez"] = (df["n_consultas_previas"] == 0).astype("int8")

# dias_desde_ultima_falta: dias entre a última falta anterior e o agendamento (NaN se nunca faltou)
df["dias_desde_ultima_falta"] = (df["t0"] - df["ultima_falta"]).dt.days

# falta_recente: categoria de quão recente foi a última falta
d = df["dias_desde_ultima_falta"]
df["falta_recente"] = pd.Categorical(
    np.select(
        [d.isna(), d <= 90, d <= 365],
        ["nunca_faltou", "ate_90d", "91_365d"],
        default="mais_de_365d",
    ),
    categories=["nunca_faltou", "ate_90d", "91_365d", "mais_de_365d"],
)


def cat_ultima(ultimo_y):
    return pd.Categorical(
        np.select(
            [ultimo_y.isna(), ultimo_y == 1],
            ["sem_previa", "faltante"],
            default="realizada",
        ),
        categories=["sem_previa", "realizada", "faltante"],
    )


# ultima_previa_mesma_especialidademae: desfecho da consulta anterior mais recente na mesma especialidade
df["ultima_previa_mesma_especialidademae"] = cat_ultima(df["ultimo_y_especialidademae"])
# ultima_previa_mesmo_motivo: idem, para o mesmo motivo (especialidade + CID3)
df["ultima_previa_mesmo_motivo"] = cat_ultima(df["ultimo_y_motivo"])

# --- 4. Variáveis de tempo e calendário --------------------------------------
# Atenção: use apenas UMA parametrização no modelo:
#   (a) dias_ate_consulta  ou  (b) diasfilaespera + antecedencia_agendamento
# dias_ate_consulta: dias entre a solicitação e a consulta
df["dias_ate_consulta"] = (df["datahora_consulta"] - df["data_solicitacao"]).dt.days
# antecedencia_agendamento: dias entre o agendamento (t0) e a consulta
df["antecedencia_agendamento"] = df["dias_ate_consulta"] - df["diasfilaespera"]
# dia_semana: dia da semana da consulta (NaN se sem data)
df["dia_semana"] = pd.Categorical(
    df["datahora_consulta"].dt.day_of_week.map(
        dict(enumerate(["seg", "ter", "qua", "qui", "sex", "sab", "dom"]))
    ),
    categories=["seg", "ter", "qua", "qui", "sex", "sab", "dom"],
)
# mes_consulta: mês da consulta (1-12)
df["mes_consulta"] = pd.Categorical(df["datahora_consulta"].dt.month, categories=range(1, 13))

df[["n_consultas_previas", "n_faltas_previas", "falta_recente", "primeira_vez"]].describe(include="all")

,n_consultas_previas,n_faltas_previas,falta_recente,primeira_vez
count,360908.000000,360908.000000,360908,360908.000000
unique,NaN,NaN,4,NaN
top,NaN,NaN,nunca_faltou,NaN
freq,NaN,NaN,322503,NaN
mean,0.543898,0.128792,NaN,0.669251
std,0.986035,0.411743,NaN,0.470484
min,0.000000,0.000000,NaN,0.000000
25%,0.000000,0.000000,NaN,0.000000
50%,0.000000,0.000000,NaN,1.000000
75%,1.000000,0.000000,NaN,1.000000


## Variaveis selecionadas 


In [ ]:
COLUNAS_FINAIS = [
    "idsolicitacao", "idpaciente", "data_solicitacao", "datahora_consulta", "rid", "t0",
    "situacaoagenda", "corregulador", "especialidademae", "cid_3dig", "cid_1dig",
    "possuiunidadeindicada", "dias_ate_consulta", "dia_semana", "mes_consulta", "idade",
    "n_consultas_previas", "n_faltas_previas", "falta_recente", "primeira_vez",
    "n_consultas_previas_especialidademae", "n_faltas_previas_especialidademae",
    "ultima_previa_mesma_especialidademae", "n_consultas_previas_motivo",
    "n_faltas_previas_motivo", "ultima_previa_mesmo_motivo", "cnes_solicitante",
    "cnesexecutante", "favela_comunidade", "taxa_alf_geral", "renda_media_winsor",
    "coordenadoria_saude", "distrito_sanitario", "status", "distancia_km",
    "tempo_caminhando", "distancia_km_trans_publico", "tempo_desloc_trans_publico",
    "quant_bus", "baldiacoes", "DS_NATUREZA_JUR",
]

df_final = df[COLUNAS_FINAIS].copy()

## 5. Gravação

In [ ]:
import os

SAIDA = "dados/processados/gercon_poa_censo_distancias_com_natureza_juridica.parquet"

os.makedirs(os.path.dirname(SAIDA), exist_ok=True)
df_final.to_parquet(SAIDA, index=False)
print(f"Gravado: {SAIDA} -> {df_final.shape[0]:,} linhas x {df_final.shape[1]} colunas")